In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

from pandas import DataFrame

from feature_extraction.extract_features import find_continuous_file_chunk_id_for_segs

In [ ]:
# Toy "segments" table
# index = global segment id (like your seg table index)
segs = pd.DataFrame(
    {
        "file": [
            np.nan,  # Missing initial segment
            "A", "A", "A",  # contiguous A
            np.nan, np.nan,  # long gap
            "A", "A",  # A again, but after gap => new group
            "B", "B",  # contiguous B
            np.nan,  # gap
            "B",  # B again, new group
            "C", "C", "C",  # contiguous C
        ],
        "start_index": [np.nan, 1, 2, 3, np.nan, np.nan, 10, 11, 0, 1, np.nan, 9, 0, 1, 2],
    },
)
print("Original:")
segs

In [ ]:
# Since NaN != NaN, gaps are filled with values that are equal
GAP = 0
assert not (segs['file'] == GAP).any()
filled = segs['file'].fillna(GAP)
ne_prev = filled != filled.shift()
group_id = ne_prev.cumsum()

segs['filled'] = filled
segs['ne_prev'] = ne_prev
segs['group_id'] = group_id
segs

In [ ]:
s = segs["file"]  # expected: Series[str | NA]
is_real = s.notna()
prev_is_real = is_real.shift(fill_value=False)

group_start = is_real & ((~prev_is_real) | (s != s.shift()))
group_id = group_start.cumsum()  # all rows get run ids
continuous_file_chunk_id = group_id.where(is_real)  # gaps -> NaN

segs['is_real'] = is_real
segs['prev_is_real'] = prev_is_real
segs['group_start'] = group_start
segs["group_id"] = group_id
segs["continuous_file_group_id"] = continuous_file_chunk_id

segs

In [ ]:
# 1) Real rows (not gap rows)
is_real = segs["file"].notna()
segs['is_real'] = is_real

# 2) Start of a new run when:
#    - current row is a gap, OR
#    - file changed vs previous row
# group_start = (~is_real) | (segs["file"] != segs["file"].shift())
group_start = segs["file"] != segs["file"].shift()
segs['group_start'] = group_start

# 3) Running id
run_id = group_start.cumsum()
segs['run_id'] = run_id

# 4) Keep ids only for real rows (gap rows stay NaN)
segs["continuous_file_group_id"] = run_id.where(is_real)

print("\nWith group id:")
segs

In [ ]:
# WTF:
print(pd.NA == pd.NA)
print(np.nan == np.nan)
print(None == None)

print()

print(pd.NA is pd.NA)
print(np.nan is np.nan)
print(None is None)

print()
GAP = object()
print(GAP == GAP)
print('<GAP>' == '<GAP>')

In [ ]:
# Equivalent to your loop in `feature_extraction/extract_features.py`
# at `extract_ptnt_features(...)`
chunks = []
for gid, g in segs[is_real].groupby("continuous_file_group_id", sort=False):
    file_name = g["file"].iloc[0]  # one file per continuous group
    first_idx = int(g["start_index"].iloc[0])
    n_segs = len(g)
    chunks.append((int(gid), file_name, first_idx, n_segs, g.index.tolist()))

print("\nExtraction chunks:")
for c in chunks:
    print(c)

# Using the Final Function

In [ ]:
edf_dir = Path('/edf_dir/')

continuous_file_chunk_id = find_continuous_file_chunk_id_for_segs(segs['file'])
continuous_file_chunk_id

In [ ]:
segs

In [ ]:
segs_chunked = segs.groupby(continuous_file_chunk_id, sort=False, dropna=True)
first_in_chunk = segs_chunked.first()

chunk_info = DataFrame({
    'start_index': first_in_chunk['start_index'],
    'file_path': edf_dir / first_in_chunk['file'],
    'n_segs': segs_chunked.size(),
},
    index=first_in_chunk.index.astype('int64'))

chunk_info

In [ ]:
for row in chunk_info.itertuples(name='FileChunkInfo'):
    print(row)
    print(type(row))
    print(row.file_path)
    print()

In [ ]:
chunk_seg_indices = {}
for chunk_id, chunk_segs in segs_chunked:
    chunk_seg_indices[chunk_id] = chunk_segs.index

chunk_seg_indices

In [ ]:
segs_chunked.indices

In [ ]:
d = {1: 'a'}
d[1.0]

In [ ]:
tuples = chunk_info.itertuples()
tuples

In [ ]:
for t in tuples:
    print(type(t))